# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import torch
from hydra.utils import instantiate

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils.notebook_setup import init_nlp_notebook # noqa: E402

# ВАЖНО: Переопределяем архитектуру на лету!
cfg = init_nlp_notebook(overrides=["model/architecture=llama3_8b"])
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Выбранная модель: {cfg.model.model_name}")

# Tokenizer

In [ ]:
# Базовый конфиг ставит padding_side: "right", но для генерации мы можем это изменить
cfg.model.tokenizer.padding_side = "left"
tokenizer = instantiate(cfg.model.tokenizer).build()

print(f"Размер словаря: {len(tokenizer):,}")
print(f"Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

# 4-bit model loading + LoRA

In [ ]:
# Записываем состояние памяти до загрузки
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated() / 1024**3

# Твой билдер сделает всю работу: скачает, квантует, добавит адаптеры
model_builder = instantiate(cfg.model.architecture.builder, tokenizer=tokenizer)
model = model_builder.build()

if torch.cuda.is_available():
    mem_after = torch.cuda.memory_allocated() / 1024**3
    print(f"\n[VRAM] Память под веса модели: {mem_after - mem_before:.2f} GB")
    print(f"[VRAM] Пиковое потребление при загрузке: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

# Params

In [ ]:
# Проверяем, действительно ли модель обернута в PEFT
from peft import PeftModel

if isinstance(model, PeftModel):
    trainable_params, all_param = model.get_nb_trainable_parameters()
    print(f"Trainable params: {trainable_params:,d}")
    print(f"All params: {all_param:,d}")
    print(f"Trainable %: {100 * trainable_params / all_param:.4f}%")
    
    # Посмотрим, куда именно прикрепились адаптеры
    # Ожидаем увидеть q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
    print("\nСтруктура слоев (первый блок трансформера):")
    for name, module in dict(model.named_modules()).items():
        if "layers.0" in name and "lora" in name.lower():
            print(f" - {name}")
else:
    print("Модель загружена БЕЗ LoRA (Full Fine-Tune режим).")

# Sanity Check

In [ ]:
from src.core.models.generation import HFTextGenerator
from src.core.models.promts import PromptManager

# Инициализируем генератор
generator = HFTextGenerator(
    model=model,
    tokenizer=tokenizer,
    generation_kwargs=cfg.model.generation_kwargs, # Берем дефолтные параметры (max_new_tokens: 256, temp: 0.7)[cite: 23]
    cleaner_cfg=cfg.model.get("cleaner") # Подключаем ResponseCleaner[cite: 23]
)

test_prompt = PromptManager.build_simple_prompt("Объясни, как работает квантование нейросетей.")

print("Начинаем генерацию. Следи за VRAM с помощью `watch -n 1 nvidia-smi` в терминале...\n")
responses = generator.generate([test_prompt])

print(f"PROMPT:\n{test_prompt}\n")
print(f"RESPONSE:\n{responses[0]}")

# Experiments with LoRA

In [ ]:
import gc

# 1. Очищаем VRAM от старой модели
del model
del model_builder
del generator
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("VRAM очищена. Меняем параметры LoRA...")

# 2. Меняем параметры прямо в объекте конфигурации Гидры
cfg.model.builder.peft_config.r = 8
cfg.model.builder.peft_config.lora_alpha = 16

# 3. Пересобираем модель с новыми параметрами
new_builder = instantiate(cfg.model.builder, tokenizer=tokenizer)
new_model = new_builder.build()

trainable_params, all_param = new_model.get_nb_trainable_parameters()
print(f"\nНовый процент обучаемых параметров (r=8): {100 * trainable_params / all_param:.4f}%")